In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
train_dir = r"F:\Education\Semester 6\Introduction to Machine Learning\Project\Dataset\Cleaned dataset\Train"
val_dir = r"F:\Education\Semester 6\Introduction to Machine Learning\Project\Dataset\Cleaned dataset\Val"


In [ ]:
IMG_SIZE = (224, 224)  # Image size for ResNet50
BATCH_SIZE = 32  # Number of images per batch

# Only rescaling (since data is already augmented)
train_datagen = ImageDataGenerator(rescale=1.0/255)
val_datagen = ImageDataGenerator(rescale=1.0/255)

# Load images from directories
train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical"
)
val_generator = val_datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical"
)


In [ ]:
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze the base model layers to keep pretrained features
base_model.trainable = False

x = Flatten()(base_model.output)
x = Dense(128, activation="relu")(x)
x = Dense(64, activation="relu")(x)
output = Dense(4, activation="softmax")(x)  # 4 output classes

# Define the final model
model = Model(inputs=base_model.input, outputs=output)

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
EPOCHS = 17
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

model.save("resnet50_brain_tumor_model_17epoches.h5")


In [ ]:
val_loss, val_acc = model.evaluate(val_generator)
print(f"Validation Accuracy: {val_acc:.2f}")
print(f"Validation Loss: {val_loss:.2f}")


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0  # Normalize pixel values
    img_array = np.expand_dims(img_array, axis=0)  # Expand dimensions for model input

    prediction = model.predict(img_array)
    class_index = np.argmax(prediction)
    
    class_labels = train_generator.class_indices
    class_labels = {v: k for k, v in class_labels.items()}  # Reverse dictionary
    
    print(f"Predicted Tumor Type: {class_labels[class_index]}")

# Example usage:
predict_image("gg (31).jpg")
predict_image("m (19).jpg")
predict_image("p (780).jpg")
predict_image("image (31).jpg")


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Get true labels and predictions
true_labels = val_generator.classes
predictions = model.predict(val_generator)
predicted_classes = np.argmax(predictions, axis=1)

# Compute confusion matrix
cm = confusion_matrix(true_labels, predicted_classes)

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=val_generator.class_indices, yticklabels=val_generator.class_indices)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
from sklearn.metrics import classification_report

report = classification_report(true_labels, predicted_classes, target_names=val_generator.class_indices)
print(report)


In [ ]:
from sklearn.metrics import roc_curve, auc

# Convert true labels to one-hot encoding
from tensorflow.keras.utils import to_categorical
true_labels_one_hot = to_categorical(true_labels, num_classes=4)

# Plot ROC Curve
plt.figure(figsize=(8, 6))
for i in range(4):  # 4 classes
    fpr, tpr, _ = roc_curve(true_labels_one_hot[:, i], predictions[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], "k--")  # Random guessing line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Extract accuracy & loss values
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)  # X-axis (Epochs)

# Plot Accuracy
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)  # First plot (Accuracy)
plt.plot(epochs, acc, 'b', label='Training Accuracy')
plt.plot(epochs, val_acc, 'r', label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training & Validation Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)  # Second plot (Loss)
plt.plot(epochs, loss, 'b', label='Training Loss')
plt.plot(epochs, val_loss, 'r', label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.legend()

plt.show()
